# 🐟 Fish Detection with YOLOv5

This notebook uses the custom-trained **YOLOv5s** model in `weights/best.pt` to detect fish in aquarium images.

The workflow is simple: **upload an image → run the custom model → draw bounding boxes → count detected fish → save the result**.


## 1. Install and load YOLOv5

The notebook downloads the YOLOv5 source code and installs its dependencies.


In [ ]:
!git clone -q https://github.com/ultralytics/yolov5.git
%cd yolov5
%pip install -qr requirements.txt

import torch
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 2. Load the trained fish detector

The trained checkpoint is stored in the repository at `weights/best.pt`.


In [ ]:
MODEL_PATH = Path("../weights/best.pt")
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

model = torch.hub.load(
    "ultralytics/yolov5",
    "custom",
    path=str(MODEL_PATH),
    force_reload=False
)

model.conf = 0.25

print("Model loaded successfully.")
print("Classes:", model.names)


## 3. Upload an aquarium image

Run this cell and choose any aquarium/fish image from your computer.


In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded.keys()))

print(f"Uploaded: {image_path}")


## 4. Detect fish and draw bounding boxes

Each detected fish is surrounded by a bounding box and labeled with the class name and confidence.


In [ ]:
def detect_fish(image_path, output_path="../results/fish_detection.jpg"):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = model(image_rgb)
    detections = results.xyxy[0].cpu().numpy()
    fish_count = 0

    for x1, y1, x2, y2, confidence, class_id in detections:
        x1, y1, x2, y2 = map(int, (x1, y1, x2, y2))
        confidence = float(confidence)
        class_id = int(class_id)
        label = model.names[class_id]
        fish_count += 1

        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image_rgb, f"{label} {confidence:.2f}",
                    (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 255, 0), 2)

    cv2.putText(image_rgb, f"Fish Detected: {fish_count}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

    plt.figure(figsize=(14, 8))
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.title(f"Fish Detection — {fish_count} fish detected")
    plt.show()

    print(f"Fish detected: {fish_count}")
    print(f"Saved result to: {output_path}")
    return detections, output_path

detections, result_path = detect_fish(image_path)


## 5. Download the annotated result


In [ ]:
from google.colab import files
files.download(str(result_path))


## 6. Training notes

The supplied training run used YOLOv5s, 60 epochs, 640×640 images, batch size 16, 444 training images, and 157 validation images. The original dataset is not included in this repository.
